# MAI 510 · Module 1 — Intent Chatbot Laboratory
## Assignments 1.1 and 1.2 · Student edition

**Atlantis University · Deep Learning**  
This notebook guides you through creating a bag-of-words classifier, evaluating **20 new sentences**, comparing **three neural networks**, and selecting a confidence threshold with a business cost argument.

**How to start:** In Google Colab, select **File → Upload notebook**, upload this `.ipynb`, and choose **Runtime → Run all** after filling in the required fields and customizing your data. **CPU is sufficient**; no GPU or paid account is required.

**Important:** The provided example is a university IT help desk. **You must change it to a domain you know firsthand** and submit your own 10 tags × 6 or more patterns and 20 genuinely new test sentences. The example is a starting point, not a submission. Dataset answers are illustrative, not institutional policies.

**Your name is required.** Every CSV, PNG, Word report, and ZIP will carry your name in the filename or report title. The ZIP contains both assignment materials.

## 1 · Student identification (REQUIRED)

In [ ]:
# EDIT these fields before running anything else.
STUDENT_NAME = ""  # Example: "Jordan Rivera" — REPLACE with your full name
STUDENT_ID = ""    # Optional; do NOT enter sensitive IDs if unnecessary
COURSE = "MAI 510 — Deep Learning"
DOMAIN = "University IT Help Desk"  # REPLACE with your first-hand domain

import re
if not STUDENT_NAME.strip() or STUDENT_NAME.strip().lower() in {"your name", "student name", "name"}:
    raise ValueError("Enter your full name in STUDENT_NAME before continuing.")
SAFE_NAME = re.sub(r"[^a-zA-Z0-9]+", "_", STUDENT_NAME.strip()).strip("_")
print(f"Student: {STUDENT_NAME} | Domain: {DOMAIN} | Report prefix: {SAFE_NAME}")

## 2 · Packages and reproducibility
Keras/TensorFlow trains the networks; NLTK tokenizes and stems text; pandas and Matplotlib create evidence. Install only if needed. Colab may need to restart its runtime after a major package change.

In [ ]:
import sys, subprocess, importlib.util
for pkg, module in [("tensorflow", "tensorflow"), ("nltk", "nltk"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("python-docx", "docx")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", pkg])
import os, json, random, io, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from nltk.stem import PorterStemmer
from nltk.tokenize import TreebankWordTokenizer
from docx import Document
from pathlib import Path
SEED=42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
OUT=Path("MAI510_"+SAFE_NAME); OUT.mkdir(exist_ok=True)
print("TensorFlow:", tf.__version__, "| Output folder:", OUT)

## 3 · Build an intents file for YOUR domain
A tag names an intent; patterns are examples of what people type; responses are what the bot may say. Assignment 1.1 requires **at least 10 unique tags and 6 patterns per tag**.

**Option A (recommended):** Edit the example `INTENTS` directly below. **Option B:** Upload an edited `intents.json` into Colab and set `USE_UPLOADED_INTENTS=True`. Do not use personally identifiable customer, patient, or student data.

In [ ]:
INTENTS = {'intents': [{'tag': 'greeting', 'patterns': ['hello', 'hi there', 'good morning', 'good afternoon', 'hey support', 'hello help desk'], 'responses': ['Hello! How can I help you?']}, {'tag': 'hours', 'patterns': ['when are you open', 'what are your hours', 'opening hours today', 'when does the office close', 'are you open on monday', 'what time do you open'], 'responses': ['Our example service desk is open weekdays from 9 AM to 5 PM.']}, {'tag': 'location', 'patterns': ['where is your office', 'how do I find you', 'what is the office address', 'where are you located', 'directions to your building', 'which building is support in'], 'responses': ['This is a demonstration; contact the service desk for its actual address.']}, {'tag': 'password_reset', 'patterns': ['I forgot my password', 'reset my login password', 'cannot remember my password', 'need a password reset', 'change forgotten password', 'help me recover my password'], 'responses': ['Use the official password reset portal or contact support.']}, {'tag': 'wifi', 'patterns': ['how do I connect to wifi', 'wireless network connection help', 'wifi is not working', 'cannot join wireless', 'what is the guest wifi', 'connect my laptop to wifi'], 'responses': ['Check the official wireless connection guide or ask the service desk.']}, {'tag': 'email', 'patterns': ['my email is not working', 'cannot access mailbox', 'help with university email', 'outlook will not open', 'email login problem', 'how to set up email'], 'responses': ['Check your account sign-in and contact support for email issues.']}, {'tag': 'printing', 'patterns': ['printer is offline', 'how can I print', 'printing does not work', 'where is the printer', 'paper jam in printer', 'cannot print my document'], 'responses': ['Check the printer status and submit a support request if necessary.']}, {'tag': 'software', 'patterns': ['install an application', 'how do I request software', 'need new software installed', 'is software available', 'request a program installation', 'can you install an app'], 'responses': ['Submit an approved software request to the service desk.']}, {'tag': 'ticket_status', 'patterns': ['check my support ticket', 'what is my ticket status', 'any update on my request', 'track my help desk case', 'has my ticket been resolved', 'follow up on support case'], 'responses': ['Use the support portal to check the status of your ticket.']}, {'tag': 'contact', 'patterns': ['how can I contact support', 'support phone number', 'talk to help desk', 'who do I call for IT', 'send a message to support', 'contact technical assistance'], 'responses': ['Contact the service desk through the official support channels.']}]}

USE_UPLOADED_INTENTS = False  # True if you uploaded intents.json to Colab
if USE_UPLOADED_INTENTS:
    with open("intents.json", encoding="utf-8") as f: INTENTS=json.load(f)
assert isinstance(INTENTS,dict) and isinstance(INTENTS.get("intents"),list)
tags=[i["tag"] for i in INTENTS["intents"]]
assert len(tags)>=10 and len(set(tags))==len(tags), "Need at least 10 UNIQUE tags"
assert all(len(i.get("patterns",[]))>=6 for i in INTENTS["intents"]), "Need at least SIX patterns per tag"
assert all(i.get("responses") for i in INTENTS["intents"]), "Every tag needs at least one response"
print("PASS:",len(tags),"tags /",sum(len(i["patterns"]) for i in INTENTS["intents"]),"training patterns")
(OUT/f"{SAFE_NAME}_intents.json").write_text(json.dumps(INTENTS,indent=2,ensure_ascii=False),encoding="utf-8")

## 4 · Transform words into numbers (tokenize → stem → bag of words)
Bag-of-words records which vocabulary stems occur. It **does not preserve word order**. Run this cell and inspect the example vector.

In [ ]:
stemmer=PorterStemmer(); tokenizer=TreebankWordTokenizer()
def tokens(text):
    return [stemmer.stem(w.lower()) for w in tokenizer.tokenize(text) if any(ch.isalnum() for ch in w)]
train_texts=[p for intent in INTENTS["intents"] for p in intent["patterns"]]
train_labels=[intent["tag"] for intent in INTENTS["intents"] for p in intent["patterns"]]
vocab=sorted(set(t for p in train_texts for t in tokens(p)))
word_index={word:i for i,word in enumerate(vocab)}
labels=sorted(tags); label_index={label:i for i,label in enumerate(labels)}
def vectorize(text):
    vec=np.zeros(len(vocab),dtype=np.float32)
    for word in set(tokens(text)):
        if word in word_index: vec[word_index[word]]=1.0
    return vec
X=np.stack([vectorize(p) for p in train_texts]); y=np.array([label_index[x] for x in train_labels])
print("Vocabulary size:",len(vocab),"Training matrix:",X.shape,"Classes:",len(labels))
print("Sample sentence:",train_texts[0]); print("Recognized stems:",[vocab[j] for j in np.flatnonzero(X[0])])
(OUT/f"{SAFE_NAME}_vocabulary.txt").write_text("\n".join(vocab),encoding="utf-8")

## 5 · Train the reference network (Assignment 1.1)
Syllabus reference: **Dense(10) → Dense(10) → Dense(number of tags, softmax)**. ReLU is used for the hidden layers; categorical labels use sparse categorical cross-entropy. Training accuracy describes the examples the network learned, **not** performance on new sentences.

In [ ]:
def make_model(hidden=(10,10)):
    tf.keras.backend.clear_session()
    model=tf.keras.Sequential([tf.keras.layers.Input(shape=(len(vocab),))])
    for units in hidden: model.add(tf.keras.layers.Dense(units,activation="relu"))
    model.add(tf.keras.layers.Dense(len(labels),activation="softmax"))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),loss="sparse_categorical_crossentropy",metrics=["accuracy"])
    return model
EPOCHS=80; BATCH_SIZE=8
model=make_model((10,10)); model.summary()
history=model.fit(X,y,epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=0,shuffle=True)
train_accuracy=float(model.evaluate(X,y,verbose=0)[1]); print("Final training accuracy:",f"{train_accuracy:.2%}")
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].plot(history.history["accuracy"]);axes[0].set(title="Training accuracy",xlabel="Epoch",ylabel="Accuracy")
axes[1].plot(history.history["loss"]);axes[1].set(title="Training loss",xlabel="Epoch",ylabel="Loss")
plt.tight_layout();plt.savefig(OUT/f"{SAFE_NAME}_training_curves.png",dpi=160);plt.show()

## 6 · Twenty unseen test sentences (Assignment 1.1)
Replace all rows below with **20 new sentences not copied from any training pattern**. Record the **expected** intent *before* running the classifier. A test sample is considered correct when the predicted label equals the expected label. Include ambiguous sentences in the set only if you can justify their expected label; label truly unresolvable items and discuss them separately.

In [ ]:
TEST_ROWS = [{'sentence': 'hey can someone assist me', 'expected_intent': 'greeting'}, {'sentence': 'good evening team', 'expected_intent': 'greeting'}, {'sentence': 'is your desk open saturday', 'expected_intent': 'hours'}, {'sentence': 'when do you shut for the day', 'expected_intent': 'hours'}, {'sentence': 'which floor do I go to', 'expected_intent': 'location'}, {'sentence': 'where should I come for assistance', 'expected_intent': 'location'}, {'sentence': 'I am locked out of my account', 'expected_intent': 'password_reset'}, {'sentence': 'could you help me get a new login secret', 'expected_intent': 'password_reset'}, {'sentence': 'the wireless keeps disconnecting', 'expected_intent': 'wifi'}, {'sentence': 'I need internet on my phone', 'expected_intent': 'wifi'}, {'sentence': 'my inbox is not loading', 'expected_intent': 'email'}, {'sentence': 'can you fix an outlook sign in error', 'expected_intent': 'email'}, {'sentence': 'my document will not come out on paper', 'expected_intent': 'printing'}, {'sentence': 'the copier has jammed again', 'expected_intent': 'printing'}, {'sentence': 'where can I get an application approved', 'expected_intent': 'software'}, {'sentence': 'I want an app put on my laptop', 'expected_intent': 'software'}, {'sentence': 'any news about the issue I submitted', 'expected_intent': 'ticket_status'}, {'sentence': 'please tell me whether my case is closed', 'expected_intent': 'ticket_status'}, {'sentence': 'can I speak with a technician', 'expected_intent': 'contact'}, {'sentence': 'what number reaches the IT team', 'expected_intent': 'contact'}]

assert len(TEST_ROWS)==20, "Exactly 20 unseen sentences are required"
assert all(r["expected_intent"] in labels for r in TEST_ROWS), "Every expected intent must be a tag"
normalized=lambda s: " ".join(s.lower().split())
training_norm={normalized(p) for p in train_texts}
assert all(normalized(r["sentence"]) not in training_norm for r in TEST_ROWS), "A test sentence duplicates a training pattern"
assert len({normalized(r["sentence"]) for r in TEST_ROWS})==20, "Test sentences must be unique"
TEST_X=np.stack([vectorize(r["sentence"]) for r in TEST_ROWS])
def evaluate_network(model):
    prob=model.predict(TEST_X,verbose=0)
    idx=np.argmax(prob,axis=1)
    return pd.DataFrame({"student_name":STUDENT_NAME,"domain":DOMAIN,"sentence":[r["sentence"] for r in TEST_ROWS],"expected_intent":[r["expected_intent"] for r in TEST_ROWS],"predicted_intent":[labels[i] for i in idx],"confidence":np.max(prob,axis=1),"correct":[labels[i]==r["expected_intent"] for i,r in zip(idx,TEST_ROWS)],"known_vocab_terms":[int(np.sum(x)) for x in TEST_X]})
results=evaluate_network(model)
print("Unseen test accuracy:", f"{results.correct.mean():.2%}")
display(results)
results.to_csv(OUT/f"{SAFE_NAME}_assignment_1_1_test_results.csv",index=False)

## 7 · Diagnose failures (Assignment 1.1)
Classify **every wrong prediction** into one primary reason: `small_vocabulary`, `overlapping_intents`, or `genuinely_ambiguous`. A classifier does not know the true reason; **your own review is required**. You may add an explanatory note. The automatically created default is `REVIEW_REQUIRED`, which must be changed before final submission if errors exist.

In [ ]:
failures=results.loc[~results.correct,["sentence","expected_intent","predicted_intent","confidence","known_vocab_terms"]].copy()
failures["failure_category"]="REVIEW_REQUIRED"  # EDIT categories by row after inspecting
failures["explanation"]="REPLACE with student analysis"
# Example for editing: failures.loc[failures.index[0], "failure_category"] = "small_vocabulary"
display(failures)
failures.to_csv(OUT/f"{SAFE_NAME}_failure_analysis_EDIT_ME.csv",index=False)
print("Failure counts:",len(failures),"| Review your CSV and written report; these categories are NOT auto-diagnosed.")

## 8 · Compare THREE architectures (Assignment 1.2)
Run models on the **same training data and the same 20 test sentences**. We use equal epoch and batch settings. Architecture A is the syllabus reference `(10,10)`; B uses one hidden layer `(10,)`; C uses two wider layers `(32,32)`. For more rigorous experiments, repeat with several seeds and compare average test results; one run is the assignment minimum.

In [ ]:
ARCHITECTURES={"reference_10_10":(10,10),"single_10":(10,),"wide_32_32":(32,32)}
models={}; comparisons=[]; variant_results={}
for name,hidden in ARCHITECTURES.items():
    tf.keras.utils.set_random_seed(SEED)
    m=make_model(hidden)
    hist=m.fit(X,y,epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=0,shuffle=True)
    train_acc=float(m.evaluate(X,y,verbose=0)[1]); res=evaluate_network(m)
    models[name]=m; variant_results[name]=res
    comparisons.append({"student_name":STUDENT_NAME,"architecture":name,"hidden_layers":str(hidden),"parameters":m.count_params(),"training_accuracy":train_acc,"unseen_test_accuracy":float(res.correct.mean()),"unseen_correct":int(res.correct.sum()),"unseen_total":len(res)})
    res.to_csv(OUT/f"{SAFE_NAME}_{name}_twenty_tests.csv",index=False)
comparison_df=pd.DataFrame(comparisons); display(comparison_df)
comparison_df.to_csv(OUT/f"{SAFE_NAME}_assignment_1_2_architecture_comparison.csv",index=False)

## 9 · Confidence threshold sweep and cost of error
When the maximum softmax probability is below the threshold, **refuse** the question rather than answer. Softmax output is *not necessarily calibrated probability*. Here we compute answered count, correct answered, wrong answered, and precision among answered. **Set the cost of a wrong answer before selecting the threshold**. A refusal has a cost too (e.g., staff follow-up); use consistent relative units.

In [ ]:
# EDIT these costs based on YOUR domain before interpreting the chart.
WRONG_ANSWER_COST=10.0     # e.g., wrong advice / rework / reputational harm
REFUSAL_COST=1.0          # e.g., user transferred to a human agent
THRESHOLDS=[0.30,0.45,0.60,0.75,0.90]  # >= five distinct values
assert WRONG_ANSWER_COST>=0 and REFUSAL_COST>=0
assert len(set(THRESHOLDS))>=5 and all(0<=t<=1 for t in THRESHOLDS)
CHOSEN_ARCHITECTURE="reference_10_10"  # EDIT after inspecting architecture results
assert CHOSEN_ARCHITECTURE in variant_results
r=variant_results[CHOSEN_ARCHITECTURE]
sweep=[]
for threshold in sorted(set(THRESHOLDS)):
    answered=r.confidence>=threshold
    correct=int((answered & r.correct).sum());wrong=int((answered & ~r.correct).sum());refused=int((~answered).sum())
    sweep.append({"student_name":STUDENT_NAME,"architecture":CHOSEN_ARCHITECTURE,"threshold":threshold,"answered":int(answered.sum()),"correct_answered":correct,"wrong_answered":wrong,"refused":refused,"accuracy_among_answered":(correct/int(answered.sum()) if answered.sum() else np.nan),"total_cost":wrong*WRONG_ANSWER_COST+refused*REFUSAL_COST})
sweep_df=pd.DataFrame(sweep);display(sweep_df)
sweep_df.to_csv(OUT/f"{SAFE_NAME}_assignment_1_2_threshold_sweep.csv",index=False)
fig,ax=plt.subplots(figsize=(9,5))
ax.plot(sweep_df.threshold,sweep_df.answered,'o-',label='Answered')
ax.plot(sweep_df.threshold,sweep_df.correct_answered,'s-',label='Correct among answered')
ax.set(title=f"Confidence threshold sweep — {STUDENT_NAME}",xlabel="Minimum confidence threshold",ylabel="Number of test sentences (out of 20)",ylim=(0,21))
ax.grid(alpha=.3);ax.legend();plt.tight_layout()
plt.savefig(OUT/f"{SAFE_NAME}_threshold_chart.png",dpi=180);plt.show()
print("Lowest example cost within tested thresholds:",sweep_df.loc[sweep_df.total_cost.idxmin(),["threshold","total_cost"]].to_dict())

## 10 · Document deployment decision and failure interpretation
**Write your reasoning rather than accepting the suggested minimum-cost value.** Your choice should consider incorrect answers, refusals, model size, how representative the 20 tests are, and whether the real-world consequences require human review.

In [ ]:
DEPLOY_THRESHOLD = 0.75  # EDIT to your recommended threshold
COST_JUSTIFICATION = "REPLACE: In my domain, a wrong answer costs ... while refusal costs ..."
ARCHITECTURE_JUSTIFICATION = "REPLACE: I would deploy ... because the 20 unseen tests show ..."
FAILURE_ANALYSIS = "REPLACE: Categorize every error as small vocabulary, overlapping intents, or genuine ambiguity. Include examples."
LIMITATIONS = "REPLACE: Note small data, unseen synonyms, overconfidence, limits of bag-of-words, and safe handoff."
assert DEPLOY_THRESHOLD>=0 and DEPLOY_THRESHOLD<=1
print("Selected:",CHOSEN_ARCHITECTURE,"@",DEPLOY_THRESHOLD)

def respond(sentence,architecture=CHOSEN_ARCHITECTURE,threshold=DEPLOY_THRESHOLD):
    probs=models[architecture].predict(vectorize(sentence).reshape(1,-1),verbose=0)[0]
    tag=labels[int(np.argmax(probs))]; conf=float(np.max(probs))
    if conf<threshold: return {"response":"I am not confident enough to answer. Please contact a human agent.","predicted_intent":tag,"confidence":conf,"refused":True}
    response=next(i["responses"][0] for i in INTENTS["intents"] if i["tag"]==tag)
    return {"response":response,"predicted_intent":tag,"confidence":conf,"refused":False}
for q in ["Could you help me?","I need assistance with my account"]: print(q,"→",respond(q))

## 11 · Generate NAMED Word reports, results and submission ZIP
After completing the editable narratives above, run this cell. It creates:
- Assignment **1.1** report with the 20-row test table and error-analysis section.
- Assignment **1.2** report with model comparison and threshold sweep; insert a one-page justification after reviewing the draft.
- Individual CSV evidence, chart images, and your custom intents JSON.
- A submission ZIP named after you.

**Required final checks:** Replace every `REPLACE` placeholder, classify the failures, verify domain/custom patterns, and **download your executed notebook separately from Colab** (`File → Download → Download .ipynb`) and add it to your submission. Files in Colab are ephemeral unless downloaded or saved to Drive.

In [ ]:
from docx.shared import Inches,Pt

def save_doc(doc,filename):
    doc.save(OUT/filename); print("Saved:",OUT/filename)
def add_table(doc,df,cols):
    t=doc.add_table(rows=1, cols=len(cols)); t.style="Table Grid"
    for c,k in zip(t.rows[0].cells,cols): c.text=k
    for _,row in df.iterrows():
        cells=t.add_row().cells
        for c,k in zip(cells,cols):
            v=row[k]; c.text=(f"{v:.3f}" if isinstance(v,(float,np.floating)) else str(v))
    return t

d1=Document();d1.add_heading("MAI 510 — Assignment 1.1",0)
d1.add_paragraph(f"Student: {STUDENT_NAME} | Domain: {DOMAIN} | Date: {pd.Timestamp.now().date()}")
d1.add_heading("Data and network",level=1)
d1.add_paragraph(f"Tags: {len(labels)} | Patterns: {len(train_texts)} | Vocabulary stems: {len(vocab)} | Reference model: Dense(10)-Dense(10)-Softmax | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Final training accuracy: {train_accuracy:.2%}.")
d1.add_heading("Unseen evaluation (20 sentences)",level=1)
d1.add_paragraph(f"Correct: {int(results.correct.sum())}/20 | Accuracy: {results.correct.mean():.2%}. These sentences must not duplicate training patterns.")
add_table(d1,results.assign(confidence=results.confidence.round(3)),["sentence","expected_intent","predicted_intent","confidence","correct"])
d1.add_heading("Failure analysis — complete before submission",level=1)
d1.add_paragraph(FAILURE_ANALYSIS)
d1.add_paragraph("Review the exported failure_analysis_EDIT_ME.csv and classify every incorrect prediction into the three required categories.")
d1.add_heading("Limitations",level=1);d1.add_paragraph(LIMITATIONS)
save_doc(d1,f"{SAFE_NAME}_Assignment_1_1_Report.docx")

d2=Document();d2.add_heading("MAI 510 — Assignment 1.2",0)
d2.add_paragraph(f"Student: {STUDENT_NAME} | Domain: {DOMAIN} | Date: {pd.Timestamp.now().date()}")
d2.add_heading("Three architecture comparison",level=1)
add_table(d2,comparison_df.assign(training_accuracy=comparison_df.training_accuracy.round(3),unseen_test_accuracy=comparison_df.unseen_test_accuracy.round(3)),["architecture","hidden_layers","parameters","training_accuracy","unseen_test_accuracy","unseen_correct"])
d2.add_heading("Confidence thresholds",level=1)
d2.add_paragraph(f"Chosen architecture: {CHOSEN_ARCHITECTURE} | Wrong-answer cost: {WRONG_ANSWER_COST} units | Refusal cost: {REFUSAL_COST} units.")
add_table(d2,sweep_df,["threshold","answered","correct_answered","wrong_answered","refused","total_cost"])
d2.add_picture(str(OUT/f"{SAFE_NAME}_threshold_chart.png"),width=Inches(6.1))
d2.add_heading("Deployment argument — complete before submission",level=1)
d2.add_paragraph(f"Chosen threshold: {DEPLOY_THRESHOLD}. {COST_JUSTIFICATION}")
d2.add_paragraph(ARCHITECTURE_JUSTIFICATION)
d2.add_paragraph(LIMITATIONS)
save_doc(d2,f"{SAFE_NAME}_Assignment_1_2_Report.docx")

manifest=("MAI 510 MODULE 1 SUBMISSION — " + STUDENT_NAME + "\n" +
          "Assignment 1.1: executed Colab notebook, custom intents.json, 2–3 page report and 20 test results.\n" +
          "Assignment 1.2: same notebook with three trained configurations, chart and one-page decision argument.\n" +
          "IMPORTANT: download your executed .ipynb separately and include it in your final LMS submission.\n" +
          "Replace all report placeholders and review failure categories; generated drafts are not final reports.\n")
(OUT/f"{SAFE_NAME}_READ_ME.txt").write_text(manifest,encoding="utf-8")
zip_path=f"MAI510_{SAFE_NAME}_Assignment_1_1_and_1_2.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for f in OUT.iterdir(): z.write(f,arcname=f.name)
print("PACKAGE READY:",zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Outside Colab: find ZIP in working directory.")

## 12 · Submission checklist
- [ ] My full name and domain are correct on both reports.
- [ ] My domain-specific intents file has **10+ tags and 6+ patterns each**.
- [ ] All **20 test sentences** are new, with expected labels and prediction/confidence results.
- [ ] My Assignment 1.1 report is **2–3 pages** (edit the generated draft as needed).
- [ ] I categorized and explained **every error** in one of the three required groups.
- [ ] I trained **3 architectures** on the same data and reported training/test outcomes.
- [ ] I swept **5+ thresholds** and included the answered/correct chart.
- [ ] I stated the **cost of wrong answers before selecting a threshold**.
- [ ] My Assignment 1.2 deployment justification is **one page** (edit the generated draft).
- [ ] I downloaded my **executed `.ipynb`** and added it to the submission ZIP or LMS upload.

**Instructor note:** This is an instructional implementation aligned to the syllabus description. It is not a byte-for-byte copy of the original reference notebook, which was not supplied.